In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from IPython.display import display
from statsmodels.stats.multitest import multipletests

from analysis.src.data import complete_case_mask
from analysis.src.io import experiment_paths, load_data
from analysis.src.plotting import configure_plotting
from analysis.src.constants import COVARIATES, OUTCOMES

In [ ]:
configure_plotting()

In [ ]:
OUT = experiment_paths('associations/primary')

DISEASES = OUTCOMES

COVS_DEMOGRAPHIC = COVARIATES[:-1]
COVS_DEPRIVATION = ['z_imd_score']
PRIMARY_POLLUTION = 'z_nox'
PRIMARY_GREENNESS = 'z_ndvi'

MODEL_SPECS = {
    'Unadjusted': [],
    '+ Demographics': COVS_DEMOGRAPHIC,
    'Primary': COVS_DEMOGRAPHIC + COVS_DEPRIVATION,
    'Primary + NOx': COVS_DEMOGRAPHIC + COVS_DEPRIVATION + [PRIMARY_POLLUTION],
    'Primary + NDVI': COVS_DEMOGRAPHIC + COVS_DEPRIVATION + [PRIMARY_GREENNESS],
    'Primary + NOx + NDVI': COVS_DEMOGRAPHIC + COVS_DEPRIVATION + [PRIMARY_POLLUTION, PRIMARY_GREENNESS]
}
ENVIRONMENTAL_MODELS = ['Primary + NOx', 'Primary + NDVI', 'Primary + NOx + NDVI']
DISPLAY_MODELS = ['Primary', *ENVIRONMENTAL_MODELS]

data = load_data()

In [ ]:
# Fit sequential adjustment models
results = []
for outcome in DISEASES:
    for label, covariates in MODEL_SPECS.items():
        model_data = data.loc[complete_case_mask(data, outcome, covariates)]
        
        formula = f'{outcome} ~ walkability_score' + (' + ' + ' + '.join(covariates) if covariates else '')
        model = smf.ols(formula, data = model_data).fit()
        ci = model.conf_int().loc['walkability_score']
        
        results.append(
            {
                'outcome': outcome,
                'model': label,
                'n_lsoas': int(model.nobs),
                'coefficient': model.params['walkability_score'],
                'ci_lower': ci.iloc[0],
                'ci_upper': ci.iloc[1],
                'p_value': model.pvalues['walkability_score'],
                'r_squared': model.rsquared
            }
        )

sequential_results = pd.DataFrame(results)
primary_coefficients = sequential_results.query("model == 'Primary'").set_index('outcome').coefficient
sequential_results['attenuation_percent'] = sequential_results.apply(
    lambda row: (
        100 * (1 - abs(row.coefficient) / abs(primary_coefficients.loc[row.outcome]))
        if row.model in ENVIRONMENTAL_MODELS
        else np.nan
    ),
    axis = 1,
)
sequential_results.to_csv(OUT.sequential_models_csv, index = False)

display(sequential_results.round(4))

In [ ]:
# Compare coefficients across adjustment models
fig, axes = plt.subplots(2, 4, figsize = (16, 8), sharey = False, constrained_layout = True)

for ax, outcome in zip(axes.flat, DISEASES):
    plot_data = sequential_results.query('outcome == @outcome').set_index('model').loc[DISPLAY_MODELS]
    positions = np.arange(len(DISPLAY_MODELS))
    
    ax.errorbar(
        positions,
        plot_data.coefficient,
        yerr = [plot_data.coefficient - plot_data.ci_lower, plot_data.ci_upper - plot_data.coefficient],
        fmt = 'o',
        capsize = 3,
        color = '#0072B2',
    )
    ax.axhline(0, color = 'black', linestyle = '--', linewidth = 0.8)
    ax.set(
        title = outcome.removeprefix('qof_'),
        xticks = positions,
        xticklabels = ['Primary', '+ NOx', '+ NDVI', '+ NOx + NDVI'],
        ylabel = 'Prevalence coefficient (95% CI)',
    )
    ax.tick_params(axis = 'x', rotation = 25, labelsize = 8)

fig.savefig(OUT.sequential_models_png, dpi = 300, bbox_inches = 'tight')

plt.show()

In [ ]:
# Correct primary outcome tests for multiple comparisons
primary = sequential_results.query("model == 'Primary'").copy()

rejected, q_values, _, _ = multipletests(primary.p_value, method = 'fdr_bh')
primary['fdr_q_value'] = q_values
primary['significant_fdr_05'] = rejected
primary.to_csv(OUT.primary_csv, index = False)

correlations = data[OUTCOMES + ['walkability_score']].corr().loc[OUTCOMES, ['walkability_score']]
correlations.rename_axis('outcome').reset_index().rename(columns = {'walkability_score': 'pearson_r'}).to_csv(
    OUT.unadjusted_correlations_csv, index = False
)

residual_rows = []
for outcome in OUTCOMES:
    sample = data.loc[complete_case_mask(data, outcome, COVARIATES)]
    model = smf.ols(f'{outcome} ~ walkability_score + ' + ' + '.join(COVARIATES), data = sample).fit()
    
    residual_rows.append(
        pd.DataFrame(
            {
                'LSOA21CD': sample.index,
                'outcome': outcome,
                'fitted': model.fittedvalues.to_numpy(),
                'residual': model.resid.to_numpy(),
                'cooks_distance': model.get_influence().cooks_distance[0]
            }
        )
    )

pd.concat(residual_rows).to_parquet(OUT.residuals_parquet, index = False)

display(primary)